In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import hashlib

def hash_value(value, hash_key = '123'):
    return hashlib.sha256((str(value) + hash_key).encode()).hexdigest()

def shift_date_unix(date):
        """
        Convert date to unix timestamp and drop the first digit
        """
        if pd.isnull(date):
            return date
        try:
            date_utc = pd.to_datetime(date).tz_localize('UTC')
            date_unix = date_utc.timestamp()
            date_unix_deid = float('0' + str(date_unix)[1:])
            return pd.to_datetime(date_unix_deid, unit = 's').strftime('%Y-%m-%d %H:%M:%S')
        except:
            return None 

def unshift_date_unix(deidentified_date):
    """
    Convert deidentified date back to original date
    """
    if pd.isnull(deidentified_date):
        return deidentified_date
    try:
        # Convert deidentified date to timestamp
        deid_timestamp = pd.to_datetime(deidentified_date).timestamp()
        
        # Remove the leading '0' and try each possible first digit (1-2)
        deid_str = str(deid_timestamp)
        if deid_str.startswith('0'):
            deid_str = deid_str[1:]  # Remove the '0'
        
        # Try first digit = 1 (for dates around 2001-2033)
        original_timestamp_1 = float('1' + deid_str)
        original_date_1 = pd.to_datetime(original_timestamp_1, unit='s')
        
        # Try first digit = 2 (for dates around 2033-2065)
        original_timestamp_2 = float('2' + deid_str)
        original_date_2 = pd.to_datetime(original_timestamp_2, unit='s')
        
        # Return the date that seems most reasonable (you may need context)
        # For now, return the one from the 2000s-2030s range
        return original_date_1.strftime('%Y-%m-%d %H:%M:%S')
        
    except:
        return None

pd.set_option('display.max_rows', 1000)

In [2]:
eroot = Path("/data/irb/surgery/pro00114885/EmoryDataset/noPHI") 

In [26]:
discharge_info = []
for year in [2016, 2017, 2018, 2019, 2020, 2021]:
    encounter_file = eroot / str(year) / f"CJSEPSIS_ENCOUNTER_{year}.dsv"
    encounter_df = pd.read_csv(encounter_file, sep = "|")
    encounter_df = encounter_df[['pat_id', 'csn', 'discharge_to']]
    discharge_info.append(encounter_df)
    
discharge_info = pd.concat(discharge_info, axis = 0)

### Read bed locations files

In [28]:
bed_labels = pd.read_csv(os.path.expandvars("$HOME/Sepy2.0/groupings/em_bed_labels.csv"))
beds_info = []
for year in [2016, 2017, 2018, 2019, 2020, 2021]:
    bed_file = eroot / str(year) / f"CJSEPSIS_BEDLOCATION_{year}.dsv"
    bed_df = pd.read_csv(bed_file, sep = "|")
    bed_df = bed_df[['pat_id', 'csn', 'bed_unit', 'bed_location_start', 'bed_location_end']]
    bed_df = bed_df.merge(
        bed_labels[['bed_units', 'icu_type']], 
        left_on='bed_unit', 
        right_on='bed_units', 
        how='left'
    )
    beds_info.append(bed_df)
    
beds_info = pd.concat(beds_info, axis = 0)

In [29]:
beds_info["bed_location_start"] = pd.to_datetime(beds_info["bed_location_start"])
mask = beds_info["icu_type"] == "sicu BEFORE 1/18/2018; cticu ON OR AFTER 1/18/2018"
cutoff_date = pd.to_datetime("1986-05-11 22:13:20")

beds_info.loc[mask, "icu_type"] = np.where(
    beds_info.loc[mask, "bed_location_start"] < cutoff_date,
    "sicu",
    "cticu"
)

mask = beds_info["icu_type"] == "cticu BEFORE 1/18/2018; micu ON OR AFTER 1/18/2018"
beds_info.loc[mask, "icu_type"] = np.where(
    beds_info.loc[mask, "bed_location_start"] < cutoff_date,
    "cticu",
    "micu"
)

mask = beds_info["icu_type"] == "sicu BEFORE 1/18/2018"
beds_info.loc[mask, "icu_type"] = np.where(
    beds_info.loc[mask, "bed_location_start"] < cutoff_date,
    "sicu",
    "other"
)

### Read vent files

In [30]:
vent_labels = pd.read_csv(os.path.expandvars("$HOME/Sepy2.0/groupings/em_vent_labels.csv"))

vent_info = []
for year in [2016, 2017, 2018]:
    vent_file = eroot / str(year) / f"CJSEPSIS_VENT_{year}.dsv"
    vent_df = pd.read_csv(vent_file, sep = "|")
    vent_df = vent_df.merge(
        vent_labels[['vent_name', 'vent_cat']], 
        left_on='vent_mode', 
        right_on='vent_name', 
        how='left'
    )
    #vent_df = vent_df[['pat_id', 'csn', 'vent_cat', 'vent_start_time', 'vent_stop_time', 'vent_rate_set']]
    vent_info.append(vent_df)
vent_info = pd.concat(vent_info, axis = 0)
vent_info['recorded_time'] = pd.to_datetime(vent_info["recorded_time"])


/tmp/ipykernel_1921244/3066789009.py:6: DtypeWarning: Columns (7,8,9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  vent_df = pd.read_csv(vent_file, sep = "|")
/tmp/ipykernel_1921244/3066789009.py:6: DtypeWarning: Columns (9,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  vent_df = pd.read_csv(vent_file, sep = "|")


In [31]:
vent_info.head()

,pat_id,csn,recorded_time,vent_start_time,vent_stop_time,vent_type,vent_mode,vent_rate_set,vent_tidal_rate_set,vent_tidal_rate_exhaled,peep,fio2,vent_name,vent_cat,Unnamed: 0
0,f5cde211701b13a02fa0efb91b631ba3196d072da9197b...,2c1343b681714024c4bc19e2abbc1b24d961a573694d9f...,1985-04-30 07:58:20,NaN,NaN,BiPAP Vision,NIPPV,NaN,NaN,NaN,NaN,0.45,NIPPV,NIPPV,NaN
1,ce73621085b580041d8aacb33036490cd8bd49fce8241f...,81640d48f608e39e3ed1e381aa28caf84c261cfa1a5c07...,1984-11-19 00:23:20,NaN,NaN,Lumis Tx,CPAP,NaN,NaN,NaN,NaN,0.28,CPAP,NIPPV,NaN
2,ce73621085b580041d8aacb33036490cd8bd49fce8241f...,81640d48f608e39e3ed1e381aa28caf84c261cfa1a5c07...,1984-11-19 23:23:20,NaN,NaN,Lumis Tx,"CPAP, NIPPV",NaN,NaN,NaN,NaN,0.28,"CPAP, NIPPV",NIPPV,NaN
3,ce73621085b580041d8aacb33036490cd8bd49fce8241f...,81640d48f608e39e3ed1e381aa28caf84c261cfa1a5c07...,1984-11-20 22:08:20,NaN,NaN,Lumis Tx,NaN,NaN,NaN,NaN,NaN,NaN,NaN,??,NaN
4,ce73621085b580041d8aacb33036490cd8bd49fce8241f...,81640d48f608e39e3ed1e381aa28caf84c261cfa1a5c07...,1984-11-21 00:15:20,NaN,NaN,Lumis Tx,"CPAP, NIPPV",NaN,NaN,NaN,NaN,0.28,"CPAP, NIPPV",NIPPV,NaN


In [32]:
inv = vent_info.loc[vent_info.vent_cat.isin(["invasive ventilation", "invasive ventilation; weaning mode"])]
inv_csn = inv.csn.unique()
len(inv_csn)

12352

### Read o2 device files

In [33]:
device_labels = pd.read_csv(os.path.expandvars("$HOME/Sepy2.0/groupings/unique_o2_devices_mapping_Jan15.csv"))
device_labels.head()

,o2_device,count,mapping
0,Room air,1186375,room_air
1,Nasal cannula,1114695,nasal_cannula
2,Trach collar,187602,trach
3,High-flow nasal cannula,129543,high_flow_nasal_cannula
4,CPAP/BiPAP Device,127344,cpap_bipap


In [34]:
device_labels = pd.read_csv(os.path.expandvars("$HOME/Sepy2.0/groupings/unique_o2_devices_mapping_Jan15.csv"))

o2_info = []
for year in [2016, 2017, 2018, 2019]:
    print(year)
    o2_file = eroot / str(year) / f"VITALS_O2_FLOW_RATE_{year}.csv"
    o2_df = pd.read_csv(o2_file, usecols = ['pat_id', 'csn', 'recorded_time', \
                                            'unassisted_resp_rate', 'o2_device', \
                                            'end_tidal_co2', 'oxygen_flow_rate'])
    o2_df['o2_device'] = o2_df.o2_device.map(dict(zip(device_labels['o2_device'], device_labels['mapping'])))
    o2_info.append(o2_df)
o2_info = pd.concat(o2_info, axis = 0)
o2_info['recorded_time'] = pd.to_datetime(o2_info['recorded_time'])

2016


/tmp/ipykernel_1921244/833099209.py:7: DtypeWarning: Columns (14,18) have mixed types. Specify dtype option on import or set low_memory=False.
  o2_df = pd.read_csv(o2_file, usecols = ['pat_id', 'csn', 'recorded_time', \


2017


/tmp/ipykernel_1921244/833099209.py:7: DtypeWarning: Columns (14,18) have mixed types. Specify dtype option on import or set low_memory=False.
  o2_df = pd.read_csv(o2_file, usecols = ['pat_id', 'csn', 'recorded_time', \


2018


/tmp/ipykernel_1921244/833099209.py:7: DtypeWarning: Columns (14,18) have mixed types. Specify dtype option on import or set low_memory=False.
  o2_df = pd.read_csv(o2_file, usecols = ['pat_id', 'csn', 'recorded_time', \


2019


/tmp/ipykernel_1921244/833099209.py:7: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  o2_df = pd.read_csv(o2_file, usecols = ['pat_id', 'csn', 'recorded_time', \


### Read ICD and CPT files

In [356]:
cpt_info = []
for year in [2016, 2018, 2019, 2020, 2021]:
    print(year)
    cpt_file = eroot / str(year) / f"CJSEPSIS_CPT_{year}.dsv"
    cpt_df = pd.read_csv(cpt_file, sep = "|")
    cpt_df = cpt_df.loc[cpt_df.procedure_cpt_cd.astype("str") == "31500"]
    #vent_df = vent_df[['pat_id', 'csn', 'vent_cat', 'vent_start_time', 'vent_stop_time', 'vent_rate_set']]
    cpt_info.append(cpt_df)
cpt_info = pd.concat(cpt_info, axis = 0)
cpt_info['procedure_dttm'] = pd.to_datetime(cpt_info['procedure_dttm'])


2016
2018
2019
2020
2021


In [357]:
icd_info = []
for year in [2016, 2018, 2019, 2020, 2021]:
    print(year)
    icd_file = eroot / str(year) / f"CJSEPSIS_ICDPROCEDURES_{year}.dsv"
    icd_df = pd.read_csv(icd_file, sep = "|")
    icd_df = icd_df.loc[icd_df.procedure_desc.str.lower().str.contains("endotracheal")]
    #vent_df = vent_df[['pat_id', 'csn', 'vent_cat', 'vent_start_time', 'vent_stop_time', 'vent_rate_set']]
    icd_info.append(icd_df)
icd_info = pd.concat(icd_info, axis = 0)
icd_info['procedure_date'] = pd.to_datetime(icd_info['procedure_date'])

2016
2018
2019
2020
2021


### CXR Labels

In [12]:
cxr_labels = pd.read_csv("/data/irb/surgery/pro00114885/EmoryDataset/RadiologyNotes/cxr_intubation_labels.csv")

/tmp/ipykernel_643247/546932732.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  cxr_labels = pd.read_csv("/data/irb/surgery/pro00114885/EmoryDataset/RadiologyNotes/cxr_intubation_labels.csv")


In [13]:
cxr_labels.DAY_VERIFIED = pd.to_datetime(cxr_labels.DAY_VERIFIED)

In [14]:
cxr_labels_p19 = cxr_labels.loc[cxr_labels.DAY_VERIFIED.apply(lambda x: x.year < 2019)]

In [15]:
cxr_labels_p19.ENCOUNTER_NBR_hashed.unique().shape, cxr_labels.ENCOUNTER_NBR_hashed.unique().shape

((18625,), (43505,))

In [16]:
cxr_labels.date_deid = pd.to_datetime(cxr_labels.date_deid)

In [17]:
del cxr_labels_p19

### Read notes

In [40]:
notes_info = []
notes_path = Path("/data/irb/surgery/pro00114885/EmoryDataset/Notes/ProcessedNotes")
for year in [2016, 2017, 2018, 2019, 2020, 2021]:
    print(year)
    notes_file = notes_path / f"notes_{year}.pkl"
    notes_df = pd.read_pickle(notes_file)
    notes_df['csn_hashed'] = notes_df['CSN'].apply(hash_value)
    notes_df = notes_df.loc[notes_df.csn_hashed.isin(vent_o2.csn.unique())]
    notes_info.append(notes_df)
notes_info = pd.concat(notes_info, axis = 0)
notes_selected = notes_info.loc[notes_info.HNAM_DOCUMENT_CLINICAL_NM.isin([
    'Airway Intubation Procedure', 'Difficult Airway / Difficult Intubation', 'Endotracheal Intubation Procedure',
'Endotracheal Intubation Procedure *ED', 'Intubation Procedure', 'Otolaryngology Consult - Airway', 'Otolaryngology Consult - Tracheostomy', 'Tracheotomy Procedure'])]
notes_selected['date_hashed'] = notes_selected["DAY_SERVICE_DESC2"].apply(shift_date_unix)

2016
2017
2018
2019
2020
2021


/tmp/ipykernel_891526/529670333.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  notes_info = pd.concat(notes_info, axis = 0)
/tmp/ipykernel_891526/529670333.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  notes_selected['date_hashed'] = notes_selected["DAY_SERVICE_DESC2"].apply(shift_date_unix)


In [19]:
del notes_info

### Merge the validation dataframes

In [29]:
cpt_info.rename(columns = {"procedure_dttm" : "recorded_time"}, inplace = True)
icd_info.rename(columns = {"procedure_date" : "recorded_time"}, inplace = True)
merged_codes = pd.merge(
    cpt_info, icd_info, \
    on = ["pat_id", "csn", "recorded_time"],
    how = "outer"
)

In [34]:
cxr_labels.rename(columns = {"date_deid" : "recorded_time"}, inplace = True)
cxr_labels.rename(columns = {"ENCOUNTER_NBR_hashed" : "csn"}, inplace = True)

In [36]:
merged_cxr = pd.merge(
    merged_codes, cxr_labels, \
    on = ["csn", "recorded_time"],
    how = "outer"
)

In [39]:
notes_selected.rename(columns = {"date_hashed" : "recorded_time"}, inplace = True)
notes_selected.rename(columns = {"csn_hashed" : "csn"}, inplace = True)


,EMPI_NBR,PATIENT_ID,CSN,HNAM_DOCUMENT_CLINICAL_NM,DAY_SERVICE_DESC2,date,year,csn,recorded_time
2201258,364924,17568818,3608086159,Intubation Procedure,06/07/2016,2016-06-07,2016.0,9a8a385525e028c3d3a2f6fb4e5e7b793a258c7c3062fc...,1984-09-28 22:13:20
2201259,916622,18120516,36878756158,Intubation Procedure,06/07/2016,2016-06-07,2016.0,0535d247cc9382edcc354e13e13f6a01335d28ac2fb06d...,1984-09-28 22:13:20
2201260,9184558,95826679,52264836146,Intubation Procedure,06/07/2016,2016-06-07,2016.0,136cee19f9ff3dbb96faa5569e32987fc7a3a40226eb1d...,1984-09-28 22:13:20
2201261,8553042,93523567,48436766352,Intubation Procedure,02/15/2017,2017-02-15,2017.0,b8d527550d05d1d0738aab0786272a613ee26f477ad071...,1985-06-08 22:13:20
2201262,223652,17427547,1406376285,Intubation Procedure,10/17/2016,2016-10-17,2016.0,3f6ff9a0a125caa98abf728b8f80b52c521678e7ab9f8a...,1985-02-07 22:13:20


In [41]:
notes_selected.recorded_time = pd.to_datetime(notes_selected.recorded_time)


In [46]:
merged_notes = pd.merge(
    merged_cxr, notes_selected, \
    on = ["csn", "recorded_time"],
    how = "outer",
    suffixes = ["", "_notes"],
)

In [47]:
merged_notes.head()

,pat_id,csn,procedure_cpt_cd,procedure_cpt_desc,modifier_cpt_cd,recorded_time,procedure_day,modifier_cpt_seq_num,group_modifier_cpt_desc,icd9_procedure_code,...,ENCNTR_ID_hashed,intubation_status,intubation_indicative_phrase,EMPI_NBR_notes,PATIENT_ID_notes,CSN,HNAM_DOCUMENT_CLINICAL_NM_notes,DAY_SERVICE_DESC2,date,year
0,NaN,0001706f6978af7975e6f5e53f7986560d71d27288d91a...,NaN,NaN,NaN,1988-07-17 22:13:20,NaN,NaN,NaN,NaN,...,a8734cc03e00a43959915d606e278dfbf0b2010fcc1973...,PRESENT,Endotracheal tube tip is in adequate position.,NaN,NaN,NaN,NaN,NaN,NaT,NaN
1,NaN,0001706f6978af7975e6f5e53f7986560d71d27288d91a...,NaN,NaN,NaN,1988-07-19 22:13:20,NaN,NaN,NaN,NaN,...,a8734cc03e00a43959915d606e278dfbf0b2010fcc1973...,NOT MENTIONED,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
2,NaN,0001706f6978af7975e6f5e53f7986560d71d27288d91a...,NaN,NaN,NaN,1988-07-22 22:13:20,NaN,NaN,NaN,NaN,...,a8734cc03e00a43959915d606e278dfbf0b2010fcc1973...,PRESENT,Support lines and tubes are adequately positio...,NaN,NaN,NaN,NaN,NaN,NaT,NaN
3,NaN,0001706f6978af7975e6f5e53f7986560d71d27288d91a...,NaN,NaN,NaN,1988-07-24 22:13:20,NaN,NaN,NaN,NaN,...,a8734cc03e00a43959915d606e278dfbf0b2010fcc1973...,PRESENT,Support lines and tubes are adequately positio...,NaN,NaN,NaN,NaN,NaN,NaT,NaN
4,NaN,0001706f6978af7975e6f5e53f7986560d71d27288d91a...,NaN,NaN,NaN,1988-07-24 22:13:20,NaN,NaN,NaN,NaN,...,a8734cc03e00a43959915d606e278dfbf0b2010fcc1973...,PRESENT,No change support apparatus.,NaN,NaN,NaN,NaN,NaN,NaT,NaN


In [48]:
merged_notes.to_pickle("/data/irb/surgery/pro00114885/EmoryDataset/vent_sessions_analysis_merged_validation.pickle")

In [35]:
merged_notes = pd.read_pickle("/data/irb/surgery/pro00114885/EmoryDataset/vent_sessions_analysis_merged_validation.pickle")

In [36]:
merged_notes.csn.unique().shape

(45287,)

### Merge Vent and O2

In [360]:
vent_bed = pd.merge(vent_info, beds_info, on = ["pat_id", "csn"], how = "left")

vent_bed["recorded_time"] = pd.to_datetime(vent_bed["recorded_time"])
vent_bed["bed_location_start"] = pd.to_datetime(vent_bed["bed_location_start"])
vent_bed["bed_location_end"] = pd.to_datetime(vent_bed["bed_location_end"])
vent_bed["correct_bed"] = (vent_bed["recorded_time"] <= vent_bed["bed_location_end"]) &  (vent_bed["recorded_time"] >= vent_bed["bed_location_start"])
vent_with_bed = vent_bed.loc[vent_bed.correct_bed]
unnamed_cols = [x for x in vent_with_bed.columns if x.startswith("Unnamed:")]
if len(unnamed_cols) > 0:
    vent_with_bed.drop(columns = unnamed_cols, inplace = True)
vent_with_bed.shape

/tmp/ipykernel_643247/2420047778.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vent_with_bed.drop(columns = unnamed_cols, inplace = True)


(586231, 20)

In [361]:
del beds_info

In [362]:
vent_o2 = pd.merge(vent_with_bed, o2_info, on = ['pat_id', 'csn', 'recorded_time'], how = 'outer')
vent_o2.shape

(26786952, 24)

In [363]:
del o2_info
del vent_with_bed

In [53]:
vent_o2.to_pickle("/data/irb/surgery/pro00114885/EmoryDataset/vent_sessions_analysis_merged_vento2.pickle")

In [82]:
vent_o2 = pd.read_pickle("/data/irb/surgery/pro00114885/EmoryDataset/vent_sessions_analysis_merged_vento2.pickle")

### ANALYSIS

In [102]:
# 1. Set to ?? to previous vent cat
mask = vent_o2.vent_cat == "??"
vent_o2.loc[mask, 'vent_cat'] = vent_o2.groupby("csn").vent_cat.transform(lambda x: x.replace("??", np.nan).ffill()).loc[mask]


/tmp/ipykernel_1921244/3463616804.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vent_o2.loc[mask, 'vent_cat'] = vent_o2.groupby("csn").vent_cat.transform(lambda x: x.replace("??", np.nan).ffill()).loc[mask]


In [121]:
# 2. Forward fill vent_cat for 12 hours to get a new column vent_cat_ffilled
#vent_o2 = vent_o2.sort_values(['csn', 'recorded_time'])

vent_o2['_ref_time'] = vent_o2['recorded_time'].where(vent_o2['vent_cat'].notna())
vent_o2['_ref_time'] = vent_o2.groupby('csn')['_ref_time'].ffill()
vent_o2['_time_diff_since_last_vent'] = vent_o2['recorded_time'] - vent_o2['_ref_time']

In [125]:
vent_o2['vent_cat_ffilled'] = vent_o2['vent_cat'].copy()
mask = vent_o2._time_diff_since_last_vent < pd.Timedelta(hours = 12)
vent_o2.loc[mask, "vent_cat_ffilled"] = vent_o2.groupby("csn")["vent_cat"].transform(lambda x: x.ffill()).loc[mask]

/tmp/ipykernel_1921244/3634469909.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vent_o2.loc[mask, "vent_cat_ffilled"] = vent_o2.groupby("csn")["vent_cat"].transform(lambda x: x.ffill()).loc[mask]


In [130]:
vent_o2.groupby('o2_device').count()

,pat_id,csn,recorded_time,vent_start_time,vent_stop_time,vent_type,vent_mode,vent_rate_set,vent_tidal_rate_set,vent_tidal_rate_exhaled,...,icu_type,correct_bed,unassisted_resp_rate,end_tidal_co2,oxygen_flow_rate,vent_cat_imputed,respiratory_support,_ref_time,_time_diff_since_last_vent,vent_cat_ffilled
o2_device,,,,,,,,,,,,,,,,,,,,,
cpap_bipap,154873,154873,154873,6464,6464,7349,6552,287,236,3070,...,4219,7549,135965,18826,18425,1005,154873,57105,57105,47290
face_mask,212485,212485,212485,300,300,341,195,44,37,120,...,195,356,190735,172741,171135,161,212485,13719,13719,5581
high_flow_nasal_cannula,225356,225356,225356,929,929,1033,210,13,9,79,...,939,1044,187093,212176,204402,835,225356,74539,74539,25250
invasive_ventilation,1477,1477,1477,37,37,20,16,32,13,15,...,4,37,1355,8,8,21,1477,414,414,414
mechanical_vent,4582,4582,4582,195,195,178,178,170,160,106,...,87,195,4118,77,77,17,4582,1947,1947,1947
nasal_cannula,3217915,3217915,3217915,5464,5464,6002,1137,43,41,310,...,4692,6034,2880184,2992599,2940220,4898,3217915,479422,479422,119955
room_air,9960329,9960329,9960329,1597,1597,1968,928,277,240,424,...,912,2050,9410757,32365,31948,1122,9960329,409316,409316,37480
trach,307563,307563,307563,3598,3598,3539,603,162,143,271,...,3399,3628,263395,227628,205240,3027,307563,107539,107539,26516


In [132]:
vent_o2['o2_device_cleaned'] = vent_o2.o2_device.copy()
mask = vent_o2.o2_device.isin(['mechanical_vent', 'invasive_ventilation', 'room_air']) & vent_o2.vent_cat_ffilled.str.contains("invasive")
vent_o2.loc[mask, 'o2_device_cleaned'] = np.nan
vent_o2.groupby('o2_device_cleaned')['csn'].count()

o2_device_cleaned
cpap_bipap                  154873
face_mask                   212485
high_flow_nasal_cannula     225356
invasive_ventilation          1074
mechanical_vent               2683
nasal_cannula              3217915
room_air                   9955777
trach                       307563
Name: csn, dtype: int64

In [133]:
# 2. Set respiratory support 
def select_resp_support(o2_device, vent_cat):
    order_of_priority = ['room_air', 'home', 'NIPPV', 'nasal_cannula', 'mechanical_vent', 'face_mask',
        'cpap_bipap', 'high_flow_nasal_cannula', 'HFNC (oxygen)', 
        'invasive ventilation', 'invasive_ventilation', 
        'invasive ventilation; weaning mode', 'trach']
    
    def get_priority(val):
        try:
            return order_of_priority.index(val)
        except (ValueError, TypeError):
            return -1
    
    if pd.isna(o2_device) and pd.notna(vent_cat):
        return vent_cat
    if pd.notna(o2_device) and pd.isna(vent_cat):
        return o2_device
    if pd.notna(o2_device) and pd.notna(vent_cat):
        # Return the one with higher priority
        return o2_device if get_priority(o2_device) > get_priority(vent_cat) else vent_cat
    return np.nan
        

In [135]:
vent_o2["respiratory_support"] = vent_o2.apply(lambda x: select_resp_support(x["o2_device_cleaned"], x["vent_cat"]), axis = 1)

In [136]:
vent_o2.groupby('respiratory_support')['csn'].nunique()

respiratory_support
HFNC (oxygen)                              1
NIPPV                                  16773
cpap_bipap                             21585
face_mask                              92853
high_flow_nasal_cannula                10537
home                                      11
invasive ventilation                   12136
invasive ventilation; weaning mode        42
invasive_ventilation                     192
mechanical_vent                          859
nasal_cannula                         200575
room_air                              893318
trach                                   7249
Name: csn, dtype: int64

In [139]:
vent_o2['_ref_time_resp_support'] = vent_o2['recorded_time'].where(vent_o2['respiratory_support'].notna())
vent_o2['_ref_time_resp_support'] = vent_o2.groupby('csn')['_ref_time_resp_support'].ffill()
vent_o2['_time_diff_since_last_resp_support'] = vent_o2['recorded_time'] - vent_o2['_ref_time_resp_support']

In [149]:
(vent_o2.loc[vent_o2.csn.isin(inv_csn)]['_time_diff_since_last_resp_support'].dt.seconds / 3600).describe()


count    4.577998e+06
mean     1.628414e+00
std      2.664367e+00
min      0.000000e+00
25%      0.000000e+00
50%      7.500000e-01
75%      2.250000e+00
max      2.398333e+01
Name: _time_diff_since_last_resp_support, dtype: float64

In [ ]:
vent_o2['respiratory_support_ffilled'] = vent_o2['respiratory_support'].copy()
mask = vent_o2._time_diff_since_last_resp_support < pd.Timedelta(hours = 12)
vent_o2.loc[mask, "respiratory_support_ffilled"] = vent_o2.groupby("csn")["respiratory_support"].transform(lambda x: x.ffill()).loc[mask]

/tmp/ipykernel_1921244/2859115659.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vent_o2.loc[mask, "respiratory_support_ffilled"] = vent_o2.groupby("csn")["respiratory_support"].transform(lambda x: x.ffill()).loc[mask]


In [114]:
mech_csns = vent_o2.loc[vent_o2.respiratory_support == "mechanical_vent"].csn.unique()
len(mech_csns)

1348

In [113]:
vent_o2.loc[vent_o2.respiratory_support == "conflict"].groupby(["o2_device", "vent_cat"]).vent_mode.count()

o2_device                vent_cat                          
cpap_bipap               NIPPV                                 6424
                         home                                     6
                         invasive ventilation                   112
face_mask                NIPPV                                  154
                         invasive ventilation                    41
high_flow_nasal_cannula  NIPPV                                  198
                         invasive ventilation                     8
invasive_ventilation     invasive ventilation                    16
mechanical_vent          NIPPV                                    0
                         invasive ventilation                   178
nasal_cannula            HFNC (oxygen)                            2
                         NIPPV                                 1095
                         home                                     1
                         invasive ventilation           

In [137]:
ventdf = vent_o2.loc[vent_o2.csn == mech_csns[0]]

In [138]:
ventdf.drop(columns = ["pat_id", "csn","vent_type", "_ref_time", "end_tidal_co2", "unassisted_resp_rate", "vent_start_time", "vent_stop_time", "bed_unit", "bed_units", "correct_bed", "vent_name", "vent_cat_imputed", "bed_location_start", "bed_location_end"]).iloc[:1000]

,recorded_time,vent_mode,vent_rate_set,vent_tidal_rate_set,vent_tidal_rate_exhaled,peep,fio2,vent_cat,icu_type,o2_device,oxygen_flow_rate,respiratory_support,_time_diff_since_last_vent,vent_cat_ffilled,o2_device_cleaned
9746,1986-01-02 06:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
9747,1986-01-02 06:38:20,AC/CMV Volume,15,450,500,6,0.6,invasive ventilation,NaN,NaN,NaN,invasive ventilation,0 days 00:00:00,invasive ventilation,NaN
9748,1986-01-02 06:43:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:05:00,invasive ventilation,NaN
9749,1986-01-02 06:58:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:20:00,invasive ventilation,NaN
9750,1986-01-02 07:13:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:35:00,invasive ventilation,NaN
9751,1986-01-02 07:31:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,room_air,NaN,NaN,0 days 00:53:00,invasive ventilation,NaN
9752,1986-01-02 08:04:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 01:26:00,invasive ventilation,NaN
9753,1986-01-02 08:05:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 01:27:00,invasive ventilation,NaN
9754,1986-01-02 09:35:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 02:57:00,invasive ventilation,NaN
9755,1986-01-02 10:04:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 03:26:00,invasive ventilation,NaN


In [96]:
test_csns = list(set(roomair_csns).intersection(set(inv_csn)))
len(test_csns)

10973

In [61]:
ventdf = vent_o2.loc[vent_o2.csn == test_csns[1500]]


In [62]:
ventdf.drop(columns = ["pat_id", "csn", "vent_start_time", "vent_stop_time", "bed_unit", "bed_units", "correct_bed", "vent_name", "bed_location_start", "bed_location_end"]).iloc[:1000]

,recorded_time,vent_type,vent_mode,vent_rate_set,vent_tidal_rate_set,vent_tidal_rate_exhaled,peep,fio2,vent_cat,icu_type,unassisted_resp_rate,o2_device,end_tidal_co2,oxygen_flow_rate
4666427,1986-03-24 15:26:20,NaN,AC/CMV Volume,24,410,NaN,15,NaN,invasive ventilation,cticu,NaN,NaN,NaN,NaN
4666428,1986-03-24 15:40:20,Hamilton Galileo,AC/CMV Volume,24,410,410,15,1.00,invasive ventilation,cticu,NaN,NaN,NaN,NaN
4666429,1986-03-24 15:43:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25,NaN,NaN,NaN
4666430,1986-03-24 15:58:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25,NaN,NaN,NaN
4666431,1986-03-24 16:19:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25,NaN,NaN,NaN
4666432,1986-03-24 16:35:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,NaN,NaN,NaN
4666433,1986-03-24 16:44:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,NaN,NaN,NaN
4666434,1986-03-24 16:54:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4666435,1986-03-24 16:59:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,NaN,NaN,NaN
4666436,1986-03-24 17:35:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25,NaN,NaN,NaN


In [7]:
vent_csns = vent_o2.csn.unique()
validation_csns = merged_notes.csn.unique()
len(validation_csns), len(vent_csns)

(45287, 915239)

In [8]:
trach = vent_o2.loc[vent_o2.o2_device == "trach"].csn.unique()
trach_cxr_notes = merged_notes.loc[merged_notes.intubation_indicative_phrase.notna() & (merged_notes.intubation_status != "NOT MENTIONED") & (merged_notes.intubation_indicative_phrase.str.lower().str.contains("collar") | merged_notes.intubation_indicative_phrase.str.lower().str.contains("tracheostomy") )].csn.unique()
trach_notes = merged_notes.loc[merged_notes.HNAM_DOCUMENT_CLINICAL_NM_notes.notna() & merged_notes.HNAM_DOCUMENT_CLINICAL_NM_notes.str.lower().str.contains("tracheo")].csn.unique()
all_trach = list(set(trach_notes).union(set(trach_cxr_notes)).union(set(trach)))

len(trach_notes), len(trach_cxr_notes),len(trach), len(all_trach)

(582, 2952, 7249, 8896)

In [9]:
validated_vent_csns = list(set(vent_csns).intersection(set(validation_csns)))
len(validated_vent_csns)


28720

In [10]:
non_validated_vent_csns = list(set(vent_csns).difference(set(validation_csns)))
len(non_validated_vent_csns)

886519

In [11]:
non_validated_on_vent = vent_o2.loc[vent_o2.csn.isin(non_validated_vent_csns) & vent_o2.vent_mode.notna() & (vent_o2.vent_cat != "NIPPV")].csn.unique()
len(non_validated_on_vent)

223

In [12]:
non_validated_trach = list(set(non_validated_on_vent).intersection(set(all_trach)))

In [13]:
non_validated_on_vent_not_trach = list(set(non_validated_on_vent).difference(set(all_trach)))

In [14]:
len(non_validated_trach), len(non_validated_on_vent_not_trach)

(34, 189)

In [53]:
asv_csns = vent_o2.loc[vent_o2.vent_mode == "Bi-Level/DuoPAP/APRV"].csn.unique()
asv_validated = list(set(asv_csns).intersection(set(validated_vent_csns)))
asv_validated_not_trach = list(set(asv_validated).difference(set(all_trach)))
len(asv_validated_not_trach), len(asv_validated), len(asv_csns)

(119, 207, 207)

In [16]:
asv_csns = vent_o2.loc[(vent_o2.vent_mode == "ASV") & (vent_o2.vent_rate_set.notna())].csn.unique()
len(asv_csns)

493

In [456]:
vent_vent_o2.sort_values(by = ["csn", "recorded_time"])

In [469]:
import tqdm
# Step 1: Create inv_ffill column
vent_o2['inv_ffill'] = np.nan

# Process each CSN separately
for csn in tqdm.tqdm(asv_csns):
    # Get indices for this CSN
    csn_mask = vent_o2['csn'] == csn
    csn_indices = vent_o2[csn_mask].index
    
    for idx in csn_indices:
        # Check if current row is AC/CMV Volume or SIMV Volume
        if vent_o2.loc[idx, 'vent_mode'] in ['AC/CMV Volume', 'SIMV Volume']:
            # Set the current row
            vent_o2.loc[idx, 'inv_ffill'] = vent_o2.loc[idx, 'vent_mode']
            
            # Get the current time
            current_time = vent_o2.loc[idx, 'recorded_time']
            cutoff_time = current_time + pd.Timedelta(hours=12)
            
            # Forward fill within same CSN for up to 12 hours OR until o2_device is not nan
            for future_idx in csn_indices[csn_indices > idx]:
                future_time = vent_o2.loc[future_idx, 'recorded_time']
                
                # Stop if past 12 hours
                if future_time > cutoff_time:
                    break
                
                # Stop if o2_device is not nan
                if pd.notna(vent_o2.loc[future_idx, 'o2_device']):
                    break
                
                # Forward fill
                vent_o2.loc[future_idx, 'inv_ffill'] = vent_o2.loc[idx, 'vent_mode']

# Step 2: Create vent_mode_imputed column
vent_o2['vent_mode_imputed'] = vent_o2['vent_mode'].copy()

# Replace ASV with inv_ffill value where inv_ffill is set
asv_and_ffill_mask = (vent_o2['vent_mode'] == 'ASV') & (pd.notna(vent_o2['inv_ffill']))
vent_o2.loc[asv_and_ffill_mask, 'vent_mode_imputed'] = vent_o2.loc[asv_and_ffill_mask, 'inv_ffill']


  0%|          | 0/3691 [00:00<?, ?it/s]/tmp/ipykernel_643247/3006077529.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'AC/CMV Volume' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  vent_o2.loc[idx, 'inv_ffill'] = vent_o2.loc[idx, 'vent_mode']
100%|██████████| 3691/3691 [1:03:44<00:00,  1.04s/it]


In [470]:
#vent_o2.to_pickle("/data/irb/surgery/pro00114885/EmoryDataset/vent_sessions_analysis_merged_vento2_imputed.pickle")

In [156]:
vent_o2_imputed = pd.read_pickle("/data/irb/surgery/pro00114885/EmoryDataset/vent_sessions_analysis_merged_vento2_imputed.pickle")

In [186]:
csns_nippv = vent_o2_imputed.loc[vent_o2_imputed.vent_cat == "NIPPV"].csn.unique()
csns_inv = vent_o2_imputed.loc[vent_o2_imputed.vent_cat.notna() & vent_o2_imputed.vent_cat.str.contains("invasive")].csn.unique()
csns_nippv_inv = list(set(csns_nippv).intersection(set(csns_inv)))
len(csns_nippv_inv)

9287

In [ ]:
vent_o2_imputed = vent_o2_imputed.sort_values(['csn', 'recorded_time']).reset_index(drop=True)


In [187]:
import tqdm
vent_o2_imputed['nippv_sus'] = False
vent_o2_imputed['recorded_time'] = pd.to_datetime(vent_o2_imputed['recorded_time'])

for csn in tqdm.tqdm(csns_nippv_inv):
    csn_data = vent_o2_imputed[vent_o2_imputed['csn'] == csn].copy()
    invasive_times = csn_data[csn_data['vent_cat'] == 'invasive_ventilation']['recorded_time'].values

    if len(invasive_times) == 0:
        continue
    
    for idx in csn_data[csn_data['vent_cat'] == 'NIPPV'].index:
        nippv_time = vent_o2_imputed.loc[idx, 'recorded_time']
        
        # Check if invasive within 6 hours before
        has_before = any((nippv_time - invasive_times > pd.Timedelta(0)) & 
                        (nippv_time - invasive_times <= pd.Timedelta(hours=6)))
        
        # Check if invasive within 6 hours after
        has_after = any((invasive_times - nippv_time > pd.Timedelta(0)) & 
                       (invasive_times - nippv_time <= pd.Timedelta(hours=6)))
        
        if has_before and has_after:
            vent_o2_imputed.loc[idx, 'nippv_sus'] = True

  4%|▍         | 412/9287 [06:54<2:28:44,  1.01s/it]


KeyboardInterrupt: 

In [211]:
asv_csns = vent_o2_imputed.loc[vent_o2_imputed.vent_mode_imputed == "ASV"].csn.unique()
asv_validated = list(set(asv_csns).intersection(set(validated_vent_csns)))
asv_validated_not_trach = list(set(asv_validated).difference(set(all_trach)))
len(asv_validated_not_trach), len(asv_validated), len(asv_csns)

(2057, 2808, 2891)

In [441]:
asv_removed = list(set(asv_csns).intersection(set(removed)))
len(asv_removed)

1305

In [250]:
csns_cpap = vent_o2_imputed.loc[vent_o2_imputed.vent_mode == "Bi-Level/DuoPAP/APRV"].csn.unique()
len(csns_cpap)

207

In [251]:
sus_nippv = vent_o2_imputed.loc[vent_o2_imputed.nippv_sus]
sus_nippv.head()

,pat_id,csn,recorded_time,vent_start_time,vent_stop_time,vent_type,vent_mode,vent_rate_set,vent_tidal_rate_set,vent_tidal_rate_exhaled,...,correct_bed,unassisted_resp_rate,o2_device,end_tidal_co2,oxygen_flow_rate,inv_ffill,_trigger_group,_trigger_value,vent_mode_imputed,nippv_sus


In [256]:
idx =60
csn = csns_cpap[idx]#"b17ebee9ec2d0d63dd55e557e0d0ddb6359c98ace5b7d9c97240cfe545a8685b"
#"90be3a2db6fe6e865915734f95c979a11b6487b373fe738ebc87f6c33a6d73e2" #"cf859564e4f7f885434ee65d3e26b8b9b5986f2a5a2e91c023f495ac8e32b342"#asv_validated[idx]
print(csn)
ventdf = vent_o2_imputed.loc[vent_o2_imputed.csn == csn]

50af0ba22c2d738888b7fd0f08eebffec0509b98efe0b21ac36356dad8c1ed5b


In [257]:
ventdf.loc[ventdf.vent_mode.notna() | ventdf.o2_device.notna()].drop(columns = ["pat_id", "csn", "vent_start_time", "vent_stop_time", "bed_unit", "bed_units", "correct_bed", "vent_name", "vent_type", "_trigger_group", "_trigger_value", "inv_ffill"]).iloc[:1000]

,recorded_time,vent_mode,vent_rate_set,vent_tidal_rate_set,vent_tidal_rate_exhaled,peep,fio2,vent_cat,bed_location_start,bed_location_end,icu_type,unassisted_resp_rate,o2_device,end_tidal_co2,oxygen_flow_rate,vent_mode_imputed,nippv_sus
8405752,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,22,nasal_cannula,3.0,3.0,NaN,False
8405753,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,22,room_air,3.0,3.0,NaN,False
8405754,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,22,nasal_cannula,3.0,3.0,NaN,False
8405755,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,22,room_air,3.0,3.0,NaN,False
8405756,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,24,nasal_cannula,3.0,3.0,NaN,False
8405757,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,24,room_air,3.0,3.0,NaN,False
8405758,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,24,nasal_cannula,3.0,3.0,NaN,False
8405759,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,24,room_air,3.0,3.0,NaN,False
8405760,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,22,nasal_cannula,3.0,3.0,NaN,False
8405761,1985-01-04 11:28:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,22,room_air,3.0,3.0,NaN,False


In [258]:
merged_notes.loc[merged_notes.csn == csn].drop(columns = ["Unnamed: 0", "notes_deid", "ENCOUNTER_ID_hashed", "ENCNTR_ID_hashed", "PATIENT_ID", "EMPI_NBR_notes", "PATIENT_ID_notes", "CSN",  "EVENT_DOCUMENT_DESC", "DAY_SERVICE_DESC2", "EVENT_DOCUMENT_KEY", "ENCOUNTER_ID","ENCOUNTER_NBR", "EMPI_NBR", "ENCNTR_ID", "HNAM_DOCUMENT_CLINICAL_ID", "HNAM_DOCUMENT_CLINICAL_NM", "DAY_VERIFIED", "DOC_TEXT", "DOC_ABSTRACT", "pat_id", "csn", "procedure_cpt_desc", "modifier_cpt_cd", "procedure_day", "modifier_cpt_seq_num", "group_modifier_cpt_desc", "procedure_desc"])

,procedure_cpt_cd,recorded_time,icd9_procedure_code,icd10_procedure_code,ACC_NBR,intubation_status,intubation_indicative_phrase,HNAM_DOCUMENT_CLINICAL_NM_notes,date,year
91128,31500,1985-01-03 22:13:20,NaN,NaN,00002DX160162086,PRESENT,Endotracheal tube at the carina directed towar...,NaN,NaT,NaN
91129,NaN,1985-01-03 22:13:20,96.04,0BH17EZ,00002DX160162086,PRESENT,Endotracheal tube at the carina directed towar...,NaN,NaT,NaN
91130,NaN,1985-01-04 22:13:20,NaN,NaN,00002DX160162208,PRESENT,ET tube is adequately positioned.,NaN,NaT,NaN
91131,NaN,1985-01-06 22:13:20,NaN,NaN,00002DX160164244,PRESENT,Endotracheal tube in adequate position.,NaN,NaT,NaN
91132,NaN,1985-01-07 22:13:20,NaN,NaN,00002DX160164838,NOT MENTIONED,NaN,NaN,NaT,NaN
91133,NaN,1985-01-07 22:13:20,NaN,NaN,00002DX160165031,NOT MENTIONED,NaN,NaN,NaT,NaN
91134,NaN,1985-01-11 22:13:20,NaN,NaN,00002DX160167378,PRESENT,"ET, NG and right subclavian catheter tips are ...",NaN,NaT,NaN


In [206]:
merged_notes.loc[merged_notes.csn == csn]["notes_deid"].values

array(['REPORT\r\rXR Chest 1 View Portable\n\nCLINICAL INDICATION: ETT placement;Other\n\nCOMPARISON: None \n\nFINDINGS: Please see Impression \n\nIMPRESSION: \nThere are no acute airspace opacities. Support apparatus is in adequate\nposition.\n\n    \n',
       'REPORT\r\rXR Chest 1 View Portable\n\nCLINICAL INDICATION: Fever\n\nCOMPARISON: {{{REDACTED-date}}} \n\nFINDINGS: Please see Impression \n\nIMPRESSION: \nSupport apparatus stable, without pneumothorax. No focal airspace opacities.\nMediastinal contours unchanged.\n\n    \n',
       'REPORT\r\rXR Chest 1 View Portable\n\nCLINICAL INDICATION: Respiratory failure\n\nCOMPARISON: {{{REDACTED-date}}} \n\nFINDINGS: Please see Impression \n\nIMPRESSION: \nThe chest is hypoinflated with atelectasis at the lung bases. There do not\nappear to be acute airspace opacities. Support apparatus is in adequate\nposition.\n\n    \n',
       'REPORT\r\rXR Chest 1 View Portable\n\nCLINICAL INDICATION: Cough\n\nCOMPARISON: {{{REDACTED-date}}} \n\nF

In [31]:
discharge_info.loc[discharge_info.csn == csn]

,pat_id,csn,discharge_to
224950,dc82da70cc09f5bc2d8f24dcd00cb7daa85a6dfb1b0a3c...,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,EXPIRED


In [37]:
ecmo.loc[ecmo.csn == csn]

,Unnamed: 0,PAT_ID,EMPI_NBR,ENCOUNTER_NBR,ENCOUNTER_ID,BED_LOCATION_START,BED_LOCATION_END,BED_UNIT,BED_ROOM,BED_ID,BED_LABEL,HOSPITAL_SERVICE,ACCOMODATION_CODE,ACCOMODATION_DESCRIPTION,ECMO cannulation,ECMO decannulation,ECLS Type,Record Number,csn


In [38]:
beds_info.loc[beds_info.csn == csn]

,pat_id,csn,bed_unit,bed_location_start,bed_location_end,bed_units,icu_type
550810,dc82da70cc09f5bc2d8f24dcd00cb7daa85a6dfb1b0a3c...,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,ED SJH,1986-08-31 11:03:20,1986-08-31 11:17:45,ED SJH,NaN
550811,dc82da70cc09f5bc2d8f24dcd00cb7daa85a6dfb1b0a3c...,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,ED SJH,1986-08-31 11:17:45,1986-08-31 17:13:30,ED SJH,NaN
550812,dc82da70cc09f5bc2d8f24dcd00cb7daa85a6dfb1b0a3c...,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,7E SJH,1986-08-31 17:13:30,1986-09-04 21:14:59,7E SJH,NaN
550813,dc82da70cc09f5bc2d8f24dcd00cb7daa85a6dfb1b0a3c...,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,2E ICU SJH,1986-09-04 21:14:59,1986-09-10 07:48:20,2E ICU SJH,msicu


In [35]:
ecmo = pd.read_csv("../ECMO_Data_matched_Jan5.csv")

In [36]:
ecmo["csn"] = ecmo.ENCOUNTER_NBR.apply(hash_value)

In [246]:

ecmo.head()

,Unnamed: 0,PAT_ID,EMPI_NBR,ENCOUNTER_NBR,ENCOUNTER_ID,BED_LOCATION_START,BED_LOCATION_END,BED_UNIT,BED_ROOM,BED_ID,BED_LABEL,HOSPITAL_SERVICE,ACCOMODATION_CODE,ACCOMODATION_DESCRIPTION,ECMO cannulation,ECMO decannulation,ECLS Type,Record Number,csn
0,0,107637578,11772301.0,70454151283,197904663,2021-10-11 04:57:30,2021-10-11 04:58:40,Main Registration EUH,Not Recorded,Not Recorded,Not Recorded,HOSPITAL MEDICINE,--,Not Recorded,2021-10-11 01:06:00,2021-11-16 11:00:00,VV,267.0,0845698fe6a9174734ac7f3d26544aa5b02bbb5dc29006...
1,1,107637578,11772301.0,70454151283,197904663,2021-10-29 13:35:44,2021-11-18 21:42:44,5G ICU EUH,G511,01,INTENSIVE CARE,HOSPITAL MEDICINE,ICU,INTENSIVE CARE,2021-10-11 01:06:00,2021-11-16 11:00:00,VV,267.0,0845698fe6a9174734ac7f3d26544aa5b02bbb5dc29006...
2,2,107637578,11772301.0,70454151283,197904663,2021-10-11 04:58:40,2021-10-29 13:35:44,5G ICU EUH,G517,01,INTENSIVE CARE,HOSPITAL MEDICINE,ICU,INTENSIVE CARE,2021-10-11 01:06:00,2021-11-16 11:00:00,VV,267.0,0845698fe6a9174734ac7f3d26544aa5b02bbb5dc29006...
3,3,107637578,11772301.0,70454151283,197904663,2021-11-18 21:42:44,2021-11-25 15:15:00,5D EUH,D547,01,PRIVATE,HOSPITAL MEDICINE,R,ROUTINE,2021-10-11 01:06:00,2021-11-16 11:00:00,VV,267.0,0845698fe6a9174734ac7f3d26544aa5b02bbb5dc29006...
4,4,107292539,11707878.0,69900791287,198213320,2021-10-14 18:51:17,2021-10-14 18:54:01,Main Registration EUH,Not Recorded,Not Recorded,Not Recorded,NEUROLOGY,--,Not Recorded,2021-08-11 12:00:00,2021-10-25 12:00:00,VV,268.0,9369951246d04d29c7a5c01d92648f1223a22e3b22a745...


In [51]:
notesdf = notes_info.loc[notes_info.csn_hashed == csn]
notesdf['date_hashed'] = notesdf["date"].apply(shift_date_unix)
notesdf

/tmp/ipykernel_891526/2533935737.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  notesdf['date_hashed'] = notesdf["date"].apply(shift_date_unix)


,EMPI_NBR,PATIENT_ID,CSN,HNAM_DOCUMENT_CLINICAL_NM,DAY_SERVICE_DESC2,date,year,csn_hashed,date_hashed
316224,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
316225,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
316226,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
316238,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
316239,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
316621,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
317418,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
317600,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
317601,8918417,94712543,50727388129,General Lab Report,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20
321322,8918417,94712543,50727388129,XR Chest 1 View Portable,05/09/2018,2018-05-09 00:00:00,2018.0,e725a9ccbb3e0ddd8c96df57770be4da7045c94281d764...,1986-08-30 22:13:20


In [ ]:
notes_selected['date_hashed'] = notes_selected["date"].apply(shift_date_unix)

In [248]:
ecmo.csn.isin(vent_o2.csn.unique()).sum()

np.int64(616)

In [61]:
vent_o2_imputed = vent_o2_imputed.sort_values(['csn', 'recorded_time'])
vent_o2_imputed['peep_filled'] = vent_o2_imputed.groupby('csn')['peep'].ffill()
vent_o2_imputed['time_since_last'] = vent_o2_imputed.groupby('csn')['recorded_time'].diff()
vent_o2_imputed.loc[vent_o2_imputed['time_since_last'] > pd.Timedelta(hours=12), 'peep_filled'] = np.nan

In [62]:
#vent_o2_imputed = vent_o2_imputed.sort_values(['csn', 'recorded_time'])
vent_o2_imputed['fio2_filled'] = vent_o2_imputed.groupby('csn')['fio2'].ffill()
#vent_o2_imputed['time_since_last'] = vent_o2_imputed.groupby('csn')['recorded_time'].diff()
vent_o2_imputed.loc[vent_o2_imputed['time_since_last'] > pd.Timedelta(hours=12), 'fio2_filled'] = np.nan

In [ ]:
vent_o2_imputed['fio2_filled'] = vent_o2_imputed.groupby('csn')['fio2'].ffill()
#vent_o2_imputed['time_since_last'] = vent_o2_imputed.groupby('csn')['recorded_time'].diff()
vent_o2_imputed.loc[vent_o2_imputed['time_since_last'] > pd.Timedelta(hours=12), 'fio2_filled'] = np.nan

In [65]:
vent_o2_imputed["vent_params_present"] = False
check = (vent_o2_imputed["peep_filled"].notna() & vent_o2_imputed["fio2_filled"].ffill().notna() & (vent_o2_imputed["vent_rate_set"].notna() | vent_o2_imputed["vent_tidal_rate_set"].notna()))
vent_o2_imputed.loc[check, "vent_params_present"] = True 

In [133]:
vent_o2_imputed["vent_cat"].unique()

array([nan, 'NIPPV', '??', 'invasive ventilation',
       'invasive ventilation; weaning mode', 'HFNC (oxygen)', 'home'],
      dtype=object)

In [137]:
vent_o2_imputed = vent_o2_imputed.sort_values(['csn', 'recorded_time'])


In [240]:
csns_nippv = vent_o2_imputed.loc[vent_o2_imputed.vent_cat == "CPAP,NIPPV"].csn.unique()
len(csns_nippv)

0

In [172]:
nippv_processed = vent_o2_imputed.loc[vent_o2_imputed.csn.isin(csns_nippv)].groupby('csn', group_keys=False).apply(check_nippv_sus)

# For non-NIPPV CSNs, set nippv_sus to False
vent_o2_imputed.loc[~vent_o2_imputed.csn.isin(csns_nippv), 'nippv_sus'] = False

# Update the NIPPV CSNs with processed values
vent_o2_imputed.loc[vent_o2_imputed.csn.isin(csns_nippv), 'nippv_sus'] = nippv_processed['nippv_sus'].values

/tmp/ipykernel_891526/858003962.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nippv_processed = vent_o2_imputed.loc[vent_o2_imputed.csn.isin(csns_nippv)].groupby('csn', group_keys=False).apply(check_nippv_sus)


In [175]:
nippv_processed['nippv_sus'].sum()

np.int64(0)

In [174]:
nippv_asv = list(set(csns_nippv).intersection(set(asv_csns)))
len(nippv_asv)

153

In [162]:
vent_o2_imputed.loc[vent_o2_imputed.csn == nippv_asv[10]].drop(columns = ["pat_id", "csn", "vent_start_time", "vent_stop_time", "bed_unit", "bed_units", "correct_bed", "vent_name", "vent_type", "_trigger_group", "_trigger_value", "inv_ffill"]).iloc[:1000]

,recorded_time,vent_mode,vent_rate_set,vent_tidal_rate_set,vent_tidal_rate_exhaled,peep,fio2,vent_cat,bed_location_start,bed_location_end,icu_type,unassisted_resp_rate,o2_device,end_tidal_co2,oxygen_flow_rate,vent_mode_imputed,nippv_sus
8884528,1985-07-01 10:48:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,18,room_air,NaN,NaN,NaN,False
8884529,1985-07-01 10:50:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,18,room_air,NaN,NaN,NaN,False
8884530,1985-07-01 15:15:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,room_air,NaN,NaN,NaN,False
8884531,1985-07-01 19:20:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,18,room_air,NaN,NaN,NaN,False
8884532,1985-07-01 19:46:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,18,room_air,NaN,NaN,NaN,False
8884533,1985-07-01 21:07:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,18,room_air,NaN,NaN,NaN,False
8884534,1985-07-01 23:21:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,nasal_cannula,2.0,2.0,NaN,False
8884535,1985-07-02 05:43:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,18,NaN,NaN,NaN,NaN,False
8884536,1985-07-02 14:13:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,18,NaN,NaN,NaN,NaN,False
8884537,1985-07-02 17:53:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,16,nasal_cannula,2.0,2.0,NaN,False


In [171]:

def check_nippv_sus(group):
    group = group.reset_index(drop=True)
    group['nippv_sus'] = False
    
    invasive_times = group[group['vent_cat'].isna() & group['vent_cat'].str.contains('invasive ventilation')]['recorded_time']
    
    for idx, row in group[group['vent_cat'] == 'NIPPV'].iterrows():
        time_diff_before = invasive_times - row['recorded_time']
        time_diff_after = row['recorded_time'] - invasive_times
        
        has_before = any((time_diff_before < pd.Timedelta(0)) & (time_diff_before >= pd.Timedelta(hours=-12)))
        has_after = any((time_diff_after < pd.Timedelta(0)) & (time_diff_after >= pd.Timedelta(hours=-12)))
        
        if has_before and has_after:
            group.loc[idx, 'nippv_sus'] = True
    
    return group



In [122]:
csns = vent_o2_imputed.loc[(vent_o2_imputed.vent_mode == "Bi-Level/DuoPAP/APRV")  ].csn.unique()
len(csns)

207

In [70]:
vent_o2_imputed.groupby("vent_mode")["vent_params_present"].agg(['mean', 'count']).sort_values(by = 'count', ascending = False)


,mean,count
vent_mode,,
AC/CMV Volume,0.992275,200383
ASV,0.016836,83927
NIPPV,0.008228,53113
CPAP+PS,0.005203,50742
APV CMV,0.979867,43114
AC Pressure,0.985830,18207
SIMV Volume,0.988092,14192
CPAP,0.003573,10916
Bi-Level/DuoPAP/APRV,0.144937,2953
